# Building a route set [Step 05.02 - Examples, centroids and separation]

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 showed the mechanism. This one is about the part you will actually
spend your time on: **choosing the example utterances**, and knowing whether the
route set you wrote is any good *before* it is in front of users.

### What you'll learn

- `max` vs `centroid` scoring, measured on the same route set.
- A **separation matrix**: how close each route sits to every other route, which
  tells you where misroutes will come from.
- What adding one example does - sometimes nothing, sometimes it fixes a route.
- Why you write a held-out set of utterances *before* you tune anything.

### Key takeaways

- Diversity of examples beats quantity. Five varied utterances outperform twenty
  paraphrases.
- The most useful diagnostic is not accuracy, it is the **margin** between the top
  route and the runner-up. Small margins are future incidents.
- Two routes that sit above ~0.7 similarity to each other should probably be one
  route with a second-stage decision inside the handler.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq  # not used in this notebook, kept for consistency


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


### The embedding model


In [ ]:
# `all-MiniLM-L6-v2` is 22M parameters, runs on CPU, and encodes a short sentence
# in single-digit milliseconds. That speed is the whole point: routing has to be
# cheap enough that it is obviously worth doing before the expensive call.

from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    """Encode a list of strings into L2-normalised vectors.

    Normalising means the dot product IS the cosine similarity, so every score
    below lives in [-1, 1] and is directly comparable.
    """
    return encoder.encode(list(texts), normalize_embeddings=True)


v = embed(["hello there", "hi!", "what is the capital of France"])
print("vector shape :", v.shape)
print("hello/hi     : %.3f" % float(v[0] @ v[1]))
print("hello/capital: %.3f" % float(v[0] @ v[2]))


### The route set


In [ ]:
# A ROUTE is: a name, a handful of EXAMPLE UTTERANCES, and a handler.
# Nothing more. There is no training step and no classifier to fit.

ROUTES = {
    "greeting": [
        "hi there",
        "hello",
        "hey, good morning",
        "yo",
        "good evening",
    ],
    "shipping_faq": [
        "how long does delivery take",
        "when will my parcel arrive",
        "do you ship internationally",
        "what are your shipping costs",
        "how fast is standard delivery",
    ],
    "order_status": [
        "where is order 1042",
        "track my order 88",
        "what happened to order number 7",
        "status of order 1042 please",
        "has order 88 shipped yet",
    ],
    "arithmetic": [
        "what is 18 times 24",
        "compute 145 plus 92",
        "calculate 900 divided by 12",
        "how much is 37 minus 19",
        "multiply 13 by 7",
    ],
}

ROUTE_NAMES = list(ROUTES)
print("%d routes, %d example utterances total"
      % (len(ROUTES), sum(len(v) for v in ROUTES.values())))


### 1. Two ways to score a route

Given a request vector `q` and a route with example vectors `E`:

- **max**:      `score = max(E @ q)`  - "does *any* example look like this?"
- **centroid**: `score = normalize(mean(E)) @ q` - "does the route's *average idea*
  look like this?"

Neither is universally better. Max is more sensitive and catches an utterance that
matches one narrow example; centroid is more stable and less likely to be dragged
around by a single odd example. Let's measure both on the same held-out set.

In [4]:
ROUTE_VECTORS = {name: embed(ex) for name, ex in ROUTES.items()}

CENTROIDS = {}
for name, vecs in ROUTE_VECTORS.items():
    c = vecs.mean(axis=0)
    CENTROIDS[name] = c / np.linalg.norm(c)      # renormalise: the mean is not unit length


def score_max(text):
    q = embed([text])[0]
    return {n: float((v @ q).max()) for n, v in ROUTE_VECTORS.items()}


def score_centroid(text):
    q = embed([text])[0]
    return {n: float(c @ q) for n, c in CENTROIDS.items()}


demo = "any idea when my delivery shows up"
print("request:", demo)
print("max      :", {k: round(v, 3) for k, v in score_max(demo).items()})
print("centroid :", {k: round(v, 3) for k, v in score_centroid(demo).items()})

request:

 any idea when my delivery shows up
max      : {'greeting': 0.167, 'shipping_faq': 0.708, 'order_status': 0.399, 'arithmetic': 0.051}


centroid : {'greeting': 0.179, 'shipping_faq': 0.618, 'order_status': 0.371, 'arithmetic': 0.041}


### 2. A held-out set, written before we tune

This is the discipline that separates a route set that works from one that only
looks like it works: write the evaluation utterances **first**, in phrasing that is
deliberately *not* copied from your examples. Include out-of-domain requests, with
gold label `None`.

In [5]:
# (utterance, gold route or None for "should fall back")
HELD_OUT = [
    ("morning!",                                   "greeting"),
    ("hiya",                                       "greeting"),
    ("good afternoon to you",                      "greeting"),
    ("do you deliver to Portugal",                 "shipping_faq"),
    ("how much do you charge for postage",         "shipping_faq"),
    ("is next-day delivery available",             "shipping_faq"),
    ("any update on order 1042",                   "order_status"),
    ("I placed order 7 last week, where is it",    "order_status"),
    ("check order number 88",                      "order_status"),
    ("what's 45 times 3",                          "arithmetic"),
    ("add 210 and 66 for me",                      "arithmetic"),
    ("divide 480 by 16",                           "arithmetic"),
    ("who wrote Pride and Prejudice",              None),
    ("summarise the theory of plate tectonics",    None),
    ("write me a haiku about rain",                None),
    ("is my data used to train your models",       None),
]
print("%d held-out utterances, %d of them out-of-domain"
      % (len(HELD_OUT), sum(1 for _, g in HELD_OUT if g is None)))

16 held-out utterances, 4 of them out-of-domain


In [6]:
def evaluate(scorer, threshold):
    """Accuracy over the held-out set for one scorer at one threshold.

    Below threshold => predict None (fall back). That is the correct answer for
    out-of-domain utterances and a miss for in-domain ones.
    """
    correct, rows = 0, []
    for text, gold in HELD_OUT:
        s = scorer(text)
        best = max(s, key=s.get)
        top = s[best]
        runner_up = sorted(s.values())[-2]
        pred = best if top >= threshold else None
        ok = (pred == gold)
        correct += ok
        rows.append((text, gold, pred, top, top - runner_up, ok))
    return correct / len(HELD_OUT), rows


for name, scorer in [("max", score_max), ("centroid", score_centroid)]:
    acc, _ = evaluate(scorer, threshold=0.45)
    print("%-9s accuracy @ threshold 0.45 : %.3f" % (name, acc))

max       accuracy @ threshold 0.45 : 0.938
centroid  accuracy @ threshold 0.45 : 0.875


### 3. Margin: the diagnostic that actually predicts incidents

Accuracy tells you what happened. The **margin** - top score minus runner-up -
tells you how nearly it went the other way. A correct route with a margin of 0.01
is a coin flip that happened to land right.

In [7]:
acc, rows = evaluate(score_max, threshold=0.45)
rows_sorted = sorted([r for r in rows if r[1] is not None], key=lambda r: r[4])

print("in-domain utterances, smallest margin first")
print("-" * 84)
print("%-40s %-13s %-13s %6s %7s" % ("utterance", "gold", "predicted", "top", "margin"))
print("-" * 84)
for text, gold, pred, top, margin, ok in rows_sorted:
    flag = " " if ok else "X"
    print("%s %-38s %-13s %-13s %6.3f %7.3f" % (flag, text[:38], gold, pred, top, margin))
print("-" * 84)
print("median margin: %.3f   smallest: %.3f"
      % (float(np.median([r[4] for r in rows_sorted])), rows_sorted[0][4]))

in-domain utterances, smallest margin first
------------------------------------------------------------------------------------
utterance                                gold          predicted        top  margin
------------------------------------------------------------------------------------
  is next-day delivery available         shipping_faq  shipping_faq   0.547   0.053
X what's 45 times 3                      arithmetic    None           0.319   0.137
  add 210 and 66 for me                  arithmetic    arithmetic     0.455   0.213
  do you deliver to Portugal             shipping_faq  shipping_faq   0.593   0.267
  I placed order 7 last week, where is i order_status  order_status   0.704   0.319
  how much do you charge for postage     shipping_faq  shipping_faq   0.653   0.367
  hiya                                   greeting      greeting       0.581   0.395
  divide 480 by 16                       arithmetic    arithmetic     0.585   0.413
  morning!                    

### 4. The separation matrix

Now look at the routes themselves rather than the traffic. For every pair of routes,
what is the highest similarity between one route's examples and another's? High
numbers here are where your misroutes will come from - regardless of what your
held-out accuracy says today.

In [8]:
print("route-to-route MAXIMUM example similarity (higher = more collision risk)")
print()
header = "%-14s" % "" + "".join("%-15s" % n for n in ROUTE_NAMES)
print(header)
for a in ROUTE_NAMES:
    row = "%-14s" % a
    for b in ROUTE_NAMES:
        if a == b:
            row += "%-15s" % "  -"
        else:
            m = float((ROUTE_VECTORS[a] @ ROUTE_VECTORS[b].T).max())
            row += "%-15s" % ("  %.3f" % m)
    print(row)

pairs = [(a, b, float((ROUTE_VECTORS[a] @ ROUTE_VECTORS[b].T).max()))
         for i, a in enumerate(ROUTE_NAMES) for b in ROUTE_NAMES[i + 1:]]
worst = max(pairs, key=lambda p: p[2])
print()
print("closest pair: %s <-> %s at %.3f" % worst)

route-to-route MAXIMUM example similarity (higher = more collision risk)

              greeting       shipping_faq   order_status   arithmetic     
greeting        -              0.178          0.205          0.160        
shipping_faq    0.178          -              0.432          0.331        
order_status    0.205          0.432          -              0.277        
arithmetic      0.160          0.331          0.277          -            

closest pair: shipping_faq <-> order_status at 0.432


`shipping_faq` and `order_status` are the close pair, and that is exactly right -
both are about parcels arriving. The difference between them is not topic, it is
whether a specific order is named. A router built on embeddings alone cannot see
that distinction reliably, which is a genuine limit of the technique, not a bug in
the route set.

The practical fix is a **cheap deterministic pre-check** in front of the router:
if the text contains an order number, it is `order_status`. Routers and rules are
not rivals.

In [9]:
import re

ORDER_RE = re.compile(r"order\s*(?:number\s*|#\s*|no\.?\s*)?(\d+)", re.I)


def route_with_rule(text, threshold=0.45):
    """Deterministic rule first, embeddings second."""
    if ORDER_RE.search(text):
        return "order_status", 1.0, "rule"
    s = score_max(text)
    best = max(s, key=s.get)
    return (best if s[best] >= threshold else None), s[best], "embedding"


hard = ["when does order 1042 arrive",
        "how long does delivery usually take",
        "will order 88 arrive before Friday"]
for h in hard:
    print("%-42s -> %s" % (h, route_with_rule(h)))

when does order 1042 arrive                -> ('order_status', 1.0, 'rule')
how long does delivery usually take        -> ('shipping_faq', 0.9855886697769165, 'embedding')
will order 88 arrive before Friday         -> ('order_status', 1.0, 'rule')


Note the middle one: no order number, so it stays with the embedding router and
lands on `shipping_faq`, which is correct. The rule only fires when it has hard
evidence. That is the right shape for a rule - **high precision, low recall,
running before the fuzzy stage**.

### 5. Does adding an example help?

The instinct when a route misfires is "add more examples". Let's test that instinct.
We add one example that is a paraphrase of an existing one, and one that covers
genuinely new phrasing, and re-measure.

In [10]:
def accuracy_with(extra_route, extra_example):
    routes = {k: list(v) for k, v in ROUTES.items()}
    routes[extra_route] = routes[extra_route] + [extra_example]
    vecs = {n: embed(ex) for n, ex in routes.items()}

    def scorer(text):
        q = embed([text])[0]
        return {n: float((v @ q).max()) for n, v in vecs.items()}

    correct = 0
    for text, gold in HELD_OUT:
        s = scorer(text)
        b = max(s, key=s.get)
        pred = b if s[b] >= 0.45 else None
        correct += (pred == gold)
    return correct / len(HELD_OUT)


base, _ = evaluate(score_max, 0.45)
para = accuracy_with("shipping_faq", "how long will delivery take")        # paraphrase
novel = accuracy_with("shipping_faq", "do you post to other countries")    # new phrasing

print("baseline (5 examples/route)                    : %.3f" % base)
print("+ paraphrase of an existing example            : %.3f" % para)
print("+ genuinely different phrasing                 : %.3f" % novel)

baseline (5 examples/route)                    : 0.938
+ paraphrase of an existing example            : 0.938
+ genuinely different phrasing                 : 0.938


Report whatever you get, including "no change". On a set this small a single
example often moves nothing, and that is the honest lesson: **route sets are tuned
by covering new phrasings, not by piling up near-duplicates**, and on a
16-utterance evaluation you cannot detect small effects at all. If you want to
tune a router seriously, you need a hundred-plus held-out utterances mined from
real traffic.

### Pitfalls

- **Tuning on the same utterances you used as examples.** Guaranteed 100%, tells
  you nothing.
- **Forgetting the out-of-domain half.** A router evaluated only on in-domain
  traffic will happily accept everything.
- **Long examples.** Keep examples the length of a real request. A paragraph-long
  example embeds to a different region than the one-liners users type.
- **Re-encoding examples per request.** Encode once at startup. If you find your
  router taking 200 ms, this is why.

### Next

Notebook 03 turns the score into a decision: thresholds, the fallback route, and
the coverage/accuracy trade-off you are implicitly choosing.